# Week 8 — Developability

**Computational Antibody Engineering**

Weeks 1–7 asked one question in many forms: *what shape is this antibody, and where does it bind?*

Today the question changes completely. A molecule can bind its target beautifully and still never become
a medicine. It aggregates in the vial. It is too viscous to inject. It expresses at 5 mg/L instead of 5
g/L. It chemically degrades on the shelf. It sticks to everything in serum and disappears in a day.

> **Which of these candidates will survive being manufactured, stored, and injected — and which should
> we kill now?**

Developability is **negative selection**: cheap filters applied early to remove molecules that will fail
expensively later. It is unglamorous, it is where most computational antibody work actually happens, and
it is the first week of this course that needs no structure prediction at all — though we will use one
anyway, because it makes the answers much better.

---
### A note on today's tools
The standard tool is the **Therapeutic Antibody Profiler (TAP)**, on the OPIG SAbPred server. Like the
epitope predictors in week 5 it is a web service, not a package, so we cannot call it from a notebook.

As in week 5, you will build the analysis yourself. What we build is **TAP-inspired, not TAP** — the
difference matters and section 7 is about exactly that. Exercise 3 sends you to the real thing.

## 0. Setup

Light week computationally. We predict six Fv structures (seconds each) and do arithmetic on them.

In [ ]:
%pip install -q anarci ImmuneBuilder openmm pdbfixer biopython pandas matplotlib
!apt-get install -qq hmmer > /dev/null 2>&1

import warnings
warnings.filterwarnings("ignore")

import re, os, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from Bio.PDB import PDBParser
from Bio.PDB.SASA import ShrakeRupley
from Bio.PDB.Polypeptide import three_to_index, index_to_one
from anarci import number
from ImmuneBuilder import ABodyBuilder2

parser = PDBParser(QUIET=True)
sasa_calc = ShrakeRupley()
STANDARD = {"ALA","ARG","ASN","ASP","CYS","GLN","GLU","GLY","HIS","ILE",
            "LEU","LYS","MET","PHE","PRO","SER","THR","TRP","TYR","VAL"}
MAX_ASA = {"ALA":129,"ARG":274,"ASN":195,"ASP":193,"CYS":167,"GLN":225,"GLU":223,
           "GLY":104,"HIS":224,"ILE":197,"LEU":201,"LYS":236,"MET":224,"PHE":240,
           "PRO":159,"SER":155,"THR":172,"TRP":285,"TYR":263,"VAL":174}
IMGT_CDR = {"CDR1": (27, 38), "CDR2": (56, 65), "CDR3": (105, 117)}
print("Ready.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Ready.


## 1. What actually kills a candidate

A binder fails development for reasons that have nothing to do with binding:

| Failure | What goes wrong | Usual structural cause |
|---|---|---|
| **Aggregation** | molecules stick to each other; immunogenic, and can fail a batch | exposed hydrophobic patches |
| **High viscosity** | cannot be injected subcutaneously at high concentration | charge asymmetry between VH and VL; charge patches |
| **Poor expression** | grams per litre become milligrams; cost of goods explodes | folding and stability problems |
| **Chemical degradation** | the molecule changes on the shelf or in the body | reactive motifs in exposed loops |
| **Polyreactivity** | binds many things weakly; cleared fast, poor exposure | large positively charged patches |

Two properties make these worth catching early:

1. **They are expensive to discover late.** Finding a viscosity problem in formulation, after two years
   of work, is a catastrophe. Finding it in a spreadsheet in week one is a Tuesday.
2. **They are partly predictable from sequence and structure** — imperfectly, but well enough to rank.

So developability screening is a **filter, not a decision**. It reorders your candidate list and tells
you where to look. The rest of today is about how much weight that ranking can carry.

## 2. The panel

Six antibodies. Three are **approved drugs** — trastuzumab, pertuzumab and bevacizumab, all of which
have been through the full development process and are injected into patients. Three are well-known
research antibodies that were never developed as therapeutics.

Keep that division in mind. It is the only calibration we will have.

In [ ]:
PANEL = {
    "trastuzumab": ("1N8Z", "B", "A", "approved"),
    "pertuzumab":  ("1S78", "D", "C", "approved"),
    "bevacizumab": ("1BJ1", "H", "L", "approved"),
    "b12":         ("1HZH", "H", "L", "research"),
    "CR6261":      ("3GBN", "H", "L", "research"),
    "murine_4G6K": ("4G6K", "H", "L", "research"),
}


def download(pdb_id):
    path = f"{pdb_id}.pdb"
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb_id}.pdb", path)
    return parser.get_structure(pdb_id, path)[0]


def chain_sequence(model, chain_id):
    return "".join(index_to_one(three_to_index(r.get_resname()))
                   if r.get_resname() in STANDARD else "X"
                   for r in model[chain_id] if r.id[0] == " ")


def variable_domain(seq):
    """Trim to the variable domain and return (sequence, IMGT position per residue)."""
    numbering, _ = number(seq, scheme="imgt")
    occupied = [(p, a) for (p, ic), a in numbering if a != "-"]
    return "".join(a for p, a in occupied), [p for p, a in occupied]


sequences = {}
for name, (pdb_id, h, l, status) in PANEL.items():
    m = download(pdb_id)
    vh, ph = variable_domain(chain_sequence(m, h))
    vl, pl = variable_domain(chain_sequence(m, l))
    sequences[name] = {"VH": vh, "VL": vl, "pos_H": ph, "pos_L": pl, "status": status}
    print(f"{name:<14}{status:<10}VH {len(vh)} aa, VL {len(vl)} aa")

trastuzumab   approved  VH 120 aa, VL 107 aa
pertuzumab    approved  VH 119 aa, VL 107 aa
bevacizumab   approved  VH 123 aa, VL 107 aa
b12           research  VH 127 aa, VL 108 aa
CR6261        research  VH 121 aa, VL 110 aa
murine_4G6K   research  VH 120 aa, VL 107 aa


## 3. The cheapest possible screen: sequence motifs

Some degradation chemistry is visible in the sequence alone, with no structure and no model. These are
short motifs where a known reaction is favoured:

| Motif | Reaction | Why it matters |
|---|---|---|
| **NG**, **NS** | asparagine deamidation | Asn → Asp changes charge; if in a CDR it can kill binding |
| **DG**, **DS** | aspartate isomerisation | backbone rearrangement, often loses activity |
| **DP** | acid-catalysed fragmentation | the molecule breaks |
| **M** | methionine oxidation | oxidised Met can destabilise or reduce potency |
| **NxS/T** | N-linked glycosylation sequon | unwanted glycan in the variable domain; heterogeneity |
| unpaired **C** | free thiol | disulfide scrambling, dimerisation, aggregation |

This is pattern matching. It takes microseconds and needs nothing but a FASTA file — which is why it is
the first thing anybody runs.

In [ ]:
MOTIFS = [
    ("deamidation",     "NG",    r"NG"),
    ("deamidation",     "NS",    r"NS"),
    ("isomerisation",   "DG",    r"DG"),
    ("isomerisation",   "DS",    r"DS"),
    ("fragmentation",   "DP",    r"DP"),
    ("oxidation",       "M",     r"M"),
    ("glycosylation",   "NxS/T", r"N[^P][ST]"),
]


def scan_sequence(seq, positions):
    """Find liability motifs. Returns one row per hit, with its IMGT position."""
    hits = []
    for kind, label, pattern in MOTIFS:
        for m in re.finditer(pattern, seq):
            i = m.start()
            p = positions[i]
            region = next((k for k, (lo, hi) in IMGT_CDR.items() if lo <= p <= hi), "framework")
            hits.append({"kind": kind, "motif": label, "imgt": p,
                         "residue": seq[i], "region": region})
    # unpaired cysteine: a variable domain should contain exactly the two that form
    # the conserved intradomain disulfide
    n_cys = seq.count("C")
    if n_cys != 2:
        hits.append({"kind": "unpaired_Cys", "motif": f"{n_cys} Cys",
                     "imgt": -1, "residue": "C", "region": "-"})
    return hits


rows = []
for name, d in sequences.items():
    for chain in ("H", "L"):
        for hit in scan_sequence(d[f"V{chain}"], d[f"pos_{chain}"]):
            rows.append({"antibody": name, "status": d["status"], "chain": chain, **hit})

liabilities = pd.DataFrame(rows)
print(f"{len(liabilities)} raw motif hits across {len(PANEL)} antibodies\n")
print("by type:")
print(liabilities.kind.value_counts().to_string())
print("\nby antibody:")
print(liabilities.groupby(["status", "antibody"]).size().to_string())

31 raw motif hits across 6 antibodies

by type:
kind
oxidation        20
deamidation       6
isomerisation     4
fragmentation     1

by antibody:
status    antibody   
approved  bevacizumab    4
          pertuzumab     5
          trastuzumab    7
research  CR6261         5
          b12            5
          murine_4G6K    5


### YOUR TURN — Question 1

1. How many raw hits did you get in total? Roughly how many per antibody?
2. Which motif type is most common? Is a hit of that type necessarily a problem?
3. Do the approved drugs have **fewer** raw hits than the research antibodies? Look before you assume.
4. Suppose you rejected any candidate with a single liability motif. How many of these six survive?
   What does that tell you about using raw motif counts as a filter?

Answer 4 before running the next section.

*Double-click and write your answers here.*

1. 31 raw motif hits across 6 antibodies. Roughly 5 hits per antibody.
2. Most common type. Oxidation, 20 of 31 — about two thirds. And no, a hit isn’t automatically a problem. An oxidation “hit” is just a Met or Trp in the sequence. Whether it actually oxidises depends on how exposed it is: a methionine buried in the Fc core is essentially inert, while the same residue sitting in a CDR loop with solvent all around it is a real risk. The motif search has no idea which one it found — it’s pattern matching on letters, with zero structural context.
3. approved = 4 + 5 + 7 = 16, research = 5 + 5 + 5 = 15. The approved drugs have more, not fewer. And trastuzumab — one of the most commercially successful antibodies ever made — has the highest count in the panel at 7.
4. Zero survive. Every one of the six has at least one motif, so a “reject anything with a liability” rule throws out bevacizumab, pertuzumab and trastuzumab along with the research antibodies. That’s the whole point of the exercise: raw motif count is not a filter. It’s a list of positions to go look at.

## 4. Why a motif is not a liability

A raw motif count is nearly useless, for one reason: **chemistry needs access.**

An NG motif buried in the hydrophobic core is not going to deamidate — the backbone is rigid, water
cannot reach it, and the local geometry does not allow the reaction. The same motif on an exposed,
flexible CDR loop is a real risk.

So we need structure. We predict each Fv (week 4's tool), compute solvent accessibility (week 5's
method), and keep only motifs that are **exposed** and **in a CDR** — where a chemical change would
both happen and matter.

In [ ]:
def fv_features(name):
    """Predict the Fv and return per-residue exposure keyed by (chain, IMGT position)."""
    path = f"dev_{name}.pdb"
    if not os.path.exists(path):
        ABodyBuilder2().predict({"H": sequences[name]["VH"],
                                 "L": sequences[name]["VL"]}).save(path)
    model = parser.get_structure("p", path)[0]
    for c in list(model):                      # strip any non-standard residues
        for r in [r for r in model[c.id] if r.id[0] != " "]:
            model[c.id].detach_child(r.id)
    sasa_calc.compute(model, level="R")

    out = {}
    for chain in ("H", "L"):
        residues = [r for r in model[chain] if r.id[0] == " "]
        positions = sequences[name][f"pos_{chain}"]
        for j, r in enumerate(residues):
            out[(chain, positions[j])] = {
                "rsa": r.sasa / MAX_ASA.get(r.get_resname(), 200),
                "aa": index_to_one(three_to_index(r.get_resname())),
            }
    return out


features = {}
for name in PANEL:
    features[name] = fv_features(name)
    print(f"  {name:<14} {len(features[name])} residues")

  trastuzumab    227 residues
  pertuzumab     226 residues
  bevacizumab    227 residues
  b12            228 residues
  CR6261         230 residues
  murine_4G6K    227 residues


In [ ]:
EXPOSED = 0.25          # RSA above this counts as solvent-accessible

liabilities["rsa"] = [
    features[r.antibody].get((r.chain, r.imgt), {}).get("rsa", np.nan)
    for r in liabilities.itertuples()
]
liabilities["exposed"] = liabilities.rsa > EXPOSED
liabilities["in_cdr"] = liabilities.region != "framework"
liabilities["high_risk"] = liabilities.exposed & liabilities.in_cdr

scored = liabilities[liabilities.imgt >= 0]
print(f"raw motif hits:              {len(scored)}")
print(f"  exposed (RSA > {EXPOSED}):        {scored.exposed.sum()}")
print(f"  in a CDR:                  {scored.in_cdr.sum()}")
print(f"  BOTH -> high risk:         {scored.high_risk.sum()}"
      f"   ({100 * scored.high_risk.sum() / len(scored):.0f}% of raw hits)\n")

high = scored[scored.high_risk].sort_values(["status", "antibody"])
print(high[["antibody", "status", "chain", "kind", "motif", "imgt", "region", "rsa"]]
      .round(2).to_string(index=False))

raw motif hits:              31
  exposed (RSA > 0.25):        9
  in a CDR:                  13
  BOTH -> high risk:         8   (26% of raw hits)

   antibody   status chain          kind motif  imgt region  rsa
 pertuzumab approved     H   deamidation    NS    59   CDR2 0.45
trastuzumab approved     H   deamidation    NG    62   CDR2 0.42
trastuzumab approved     H isomerisation    DG   110   CDR3 0.55
     CR6261 research     H     oxidation     M   108   CDR3 0.36
        b12 research     H   deamidation    NG    62   CDR2 0.50
murine_4G6K research     H isomerisation    DG    59   CDR2 0.48
murine_4G6K research     H fragmentation    DP   110   CDR3 0.45
murine_4G6K research     L     oxidation     M   109   CDR3 0.39


### The filter did most of the work

Structure removed roughly three quarters of the raw hits. Most motifs in an antibody are buried, sit in
the framework, or both — they are sequence coincidences, not liabilities.

This is the shape of a good screen: **cheap test first, expensive test only on the survivors.** The
regex costs nothing and runs on a million sequences; the structural check costs seconds per candidate
and is applied only to hits.

Now look at *which* antibodies carry the surviving high-risk liabilities.

## 5. The result that should stop you

Look carefully at the high-risk table above, specifically at the approved drugs.

**Trastuzumab carries two high-risk liabilities**: an exposed NG deamidation motif in CDR-H2, and an
exposed DG isomerisation motif in CDR-H3. Both are in CDRs. Both are solvent accessible. By the filter
we just built, trastuzumab looks like a candidate you might kill.

Trastuzumab is one of the most commercially successful antibodies ever made.

**If your developability filter would have killed trastuzumab, your filter is wrong.**

Why does a real drug carry real liabilities?

- **A liability is a rate, not a switch.** The chemistry happens slowly. Whether it matters depends on
  how fast, under what storage conditions, and whether the altered molecule loses potency.
- **Formulation exists.** pH, excipients, temperature and packaging all control degradation. Problems
  that computation flags are routinely *managed* rather than engineered out.
- **Liabilities trade against binding.** The exposed CDR residues that react are frequently the residues
  doing the binding. Removing the liability can remove the drug.

So the correct use of this output is: **a ranked list of things to measure**, not a kill list. The
decision to kill a candidate needs experimental data — a forced-degradation study, an accelerated
stability run — and this analysis tells you which candidates and which residues to spend that money on.

## 6. Biophysical risk: TAP-inspired metrics

Chemical liabilities are one axis. The other is **biophysical**: aggregation and viscosity, driven by
the size and character of surface patches on the Fv.

TAP reports five metrics. We compute our own versions from the predicted structures:

| Our metric | Proxy for | Risk direction |
|---|---|---|
| **CDR length** | total residues across all six CDRs | longer → more surface, more risk |
| **PSH** | exposed hydrophobic surface | higher → aggregation risk |
| **PPC** | positive charge patches near CDRs | higher → polyreactivity, fast clearance |
| **PNC** | negative charge patches near CDRs | higher → risk |
| **SFvCSP** | VH/VL charge symmetry (product of net charges) | large magnitude → viscosity risk |

These are simplified. Real TAP uses carefully defined patch geometry and, crucially, a **reference
distribution** we do not have. Section 7 is about that gap.

In [ ]:
KYTE_DOOLITTLE = {"A":1.8,"R":-4.5,"N":-3.5,"D":-3.5,"C":2.5,"Q":-3.5,"E":-3.5,"G":-0.4,
                  "H":-3.2,"I":4.5,"L":3.8,"K":-3.9,"M":1.9,"F":2.8,"P":-1.6,"S":-0.8,
                  "T":-0.7,"W":-0.9,"Y":-1.3,"V":4.2}
CHARGE = {"D": -1, "E": -1, "K": 1, "R": 1, "H": 0.1}


def developability_metrics(name):
    f = features[name]
    annotated = {}
    for (chain, p), v in f.items():
        region = next((k for k, (lo, hi) in IMGT_CDR.items() if lo <= p <= hi), "framework")
        annotated[(chain, p)] = {**v, "region": region}

    cdr_length = sum(1 for v in annotated.values() if v["region"] != "framework")
    surface = [v for v in annotated.values() if v["rsa"] > 0.15]

    psh = sum(KYTE_DOOLITTLE[v["aa"]] * v["rsa"]
              for v in surface if KYTE_DOOLITTLE[v["aa"]] > 0)
    ppc = sum(v["rsa"] for v in surface
              if CHARGE.get(v["aa"], 0) > 0.5 and v["region"] != "framework")
    pnc = sum(v["rsa"] for v in surface
              if CHARGE.get(v["aa"], 0) < 0 and v["region"] != "framework")
    q_h = sum(CHARGE.get(v["aa"], 0) for (c, p), v in annotated.items() if c == "H")
    q_l = sum(CHARGE.get(v["aa"], 0) for (c, p), v in annotated.items() if c == "L")

    return {"antibody": name, "status": sequences[name]["status"],
            "CDR_len": cdr_length, "PSH": round(psh, 1),
            "PPC": round(ppc, 2), "PNC": round(pnc, 2),
            "SFvCSP": round(q_h * q_l, 1)}


metrics = pd.DataFrame([developability_metrics(n) for n in PANEL])
metrics

,antibody,status,CDR_len,PSH,PPC,PNC,SFvCSP
0,trastuzumab,approved,47,18.4,0.49,1.33,2.3
1,pertuzumab,approved,46,22.2,0.00,0.86,0.0
2,bevacizumab,approved,47,22.5,0.00,0.95,0.0
3,b12,research,48,20.7,2.32,0.20,21.7
4,CR6261,research,52,24.2,1.78,0.42,4.2
5,murine_4G6K,research,47,18.2,0.78,1.93,3.4


In [ ]:
METRIC_COLS = ["CDR_len", "PSH", "PPC", "PNC", "SFvCSP"]

ranks = metrics[["antibody", "status"]].copy()
for c in METRIC_COLS:
    ranks[c] = metrics[c].rank(ascending=False).astype(int)   # 1 = riskiest
ranks["mean_rank"] = ranks[METRIC_COLS].mean(axis=1).round(1)
ranks = ranks.sort_values("mean_rank")

print("Rank on each metric (1 = riskiest):\n")
print(ranks.to_string(index=False))

spread = ranks[METRIC_COLS].max(axis=1) - ranks[METRIC_COLS].min(axis=1)
print("\nHow much does each antibody's rank move between metrics?")
for ab, s in zip(ranks.antibody, spread):
    print(f"   {ab:<14} spans {s} rank positions")

Rank on each metric (1 = riskiest):

   antibody   status  CDR_len  PSH  PPC  PNC  SFvCSP  mean_rank
     CR6261 research        1    1    2    5       2        2.2
        b12 research        2    4    1    6       1        2.8
murine_4G6K research        4    6    3    1       3        3.4
trastuzumab approved        4    5    4    2       4        3.8
bevacizumab approved        4    2    5    3       5        3.8
 pertuzumab approved        6    3    5    4       5        4.6

How much does each antibody's rank move between metrics?
   CR6261         spans 4 rank positions
   b12            spans 5 rank positions
   murine_4G6K    spans 5 rank positions
   trastuzumab    spans 3 rank positions
   bevacizumab    spans 3 rank positions
   pertuzumab     spans 3 rank positions


### YOUR TURN — Question 2

1. Which antibody is riskiest by **mean rank**? Which is safest?
2. Find an antibody whose rank **moves a lot** between metrics. Name the metric where it looks risky and
   the one where it looks safe. What does that do to the idea of a single developability score?
3. Do the three approved drugs rank systematically safer than the three research antibodies? Quote mean
   ranks. Is the separation clean?
4. **The decision.** You must kill two candidates. Name them, and justify the choice using both the
   chemical liabilities (section 4) and the biophysical metrics. Then state explicitly what evidence
   would change your mind.

*Double-click and write your answers here.*

1. CR6261 is riskiest, pertuzumab is safest.
2. b12; PPC is risky PNC is safer. There might not be a single developability score.
3. yes. approved ones mean ranks are further away from 1.
4. b12 and CR6261 because B12 moves a lot between metrics and both of them are risky.

### Why the metrics disagree

They measure genuinely different failure modes. An antibody with a large hydrophobic patch and no charge
asymmetry has an aggregation risk and no viscosity risk. Averaging those into one number destroys the
information you needed.

This is why TAP reports five values rather than a score, and why the output of a developability screen
should be read as a **profile**, not a grade. "Which candidate is most developable?" is usually the
wrong question. "Which failure mode is each candidate exposed to, and which of those can we manage?" is
the right one.

## 7. What we have not done

Be clear about the limits of what you just built, because the gap between it and a real tool is
instructive.

**We have no reference distribution.** This is the big one. TAP's value is not that it computes five
numbers — it is that it places those numbers within the distribution of **754 post-Phase-I therapeutic
Fv domains**, so "unusual" is defined against molecules that demonstrably made it through development.
Our reference set is **three approved drugs**. Three is not a distribution. We can rank our panel
against itself; we cannot say whether any value is unusual for a therapeutic.

**Our metrics are simplified.** Real patch analysis uses proper surface geometry, not a sum over
residues weighted by accessibility.

**We used predicted structures.** Week 4: framework good, CDR loops less so — and these metrics are
dominated by CDR surface. Week 5 showed the same analysis on a predicted structure can shift which
specific residues you nominate.

**We ignored the constant domains.** Aggregation and viscosity are properties of the whole molecule.

**And we have measured no biology at all.** Every number today is a proxy. The actual quantities —
aggregation onset temperature, viscosity at 150 mg/mL, expression titre, serum half-life — are measured
in a laboratory. Developability prediction schedules those experiments; it does not replace them.

## 8. Deliverable

**Submit:** this notebook executed top to bottom, committed as `week08/week08_developability.ipynb`.

Must contain: the raw and filtered liability tables, the metrics table and rank comparison, your answers
to Questions 1 and 2, and the report below.

In [ ]:
print("=" * 66)
print("WEEK 8 SUMMARY - developability screen")
print("=" * 66)
print(f"Panel: {len(PANEL)} antibodies "
      f"({sum(1 for d in sequences.values() if d['status']=='approved')} approved, "
      f"{sum(1 for d in sequences.values() if d['status']=='research')} research)")
print()
print(f"Raw sequence motif hits:      {len(scored)}")
print(f"High risk (exposed + in CDR): {scored.high_risk.sum()}"
      f"  ({100*scored.high_risk.sum()/len(scored):.0f}% survive the structural filter)")
print()
print("High-risk liabilities per antibody:")
for ab in metrics.antibody:
    n = int(scored[(scored.antibody == ab) & scored.high_risk].shape[0])
    print(f"   {ab:<14}{n}   ({sequences[ab]['status']})")
print()
print("Riskiest by mean rank: ", ranks.iloc[0].antibody)
print("Safest by mean rank:   ", ranks.iloc[-1].antibody)
print("=" * 66)

WEEK 8 SUMMARY - developability screen
Panel: 6 antibodies (3 approved, 3 research)

Raw sequence motif hits:      31
High risk (exposed + in CDR): 8  (26% survive the structural filter)

High-risk liabilities per antibody:
   trastuzumab   2   (approved)
   pertuzumab    1   (approved)
   bevacizumab   0   (approved)
   b12           1   (research)
   CR6261        1   (research)
   murine_4G6K   3   (research)

Riskiest by mean rank:  CR6261
Safest by mean rank:    pertuzumab


### Your report

*Double-click and write 250–350 words.*

Write the developability section of a candidate-selection memo. Your reader is a project team choosing
which two of six candidates to drop.

Cover:

- your method in one short paragraph, including the exposure threshold and why a structural filter was
  applied to the motif hits
- the two candidates you would drop, with reasons drawn from **both** analyses
- the candidate you are **least** sure about, and why
- what the trastuzumab result means for how the team should read your ranking
- the single experiment you would run next, and on which candidate

One thing to get right: you are advising on **risk**, and risk is not disqualification. A memo that
reads "candidates X and Y have liabilities, therefore kill them" has not understood section 5. A memo
that says "I would not kill anything, everything is uncertain" is equally useless — you were asked for a
recommendation. Make one, and say what would overturn it.

---

**Your answer:**
I scanned all six sequences for short motifs where known degradation chemistry happens — deamidation, isomerisation, fragmentation, oxidation, glycosylation — which gave 31 hits, far too many to act on, since rejecting any candidate with a single hit would have left us with none. So I filtered them with structure: a motif only counts if it is on the surface (RSA > 0.25) and sits in a CDR, because buried motifs cannot react and framework motifs do not affect binding. Eight survived. I also ranked the six on five biophysical measures: CDR length, exposed hydrophobic surface, positive and negative charge patches, and VH/VL charge balance. On that basis I would drop murine 4G6K and CR6261. 4G6K has three surviving liabilities, the most in the panel, and they are the worst kinds — a fragmentation site in CDR-H3, an isomerisation site in CDR-H2, and an exposed methionine in CDR-L3 — all in binding loops, so fixing them risks affinity, and a site where the molecule breaks is not something buffer can manage. CR6261 looks worst biophysically (mean rank 2.2), with the longest CDRs and the most exposed hydrophobic surface of the six, which is an aggregation profile, plus one exposed methionine in CDR-H3. I am least sure about b12: it ranks second-riskiest and is worst in the panel on both positive charge patch and charge asymmetry, so the polyreactivity and viscosity concern is real, but its only chemical liability is an exposed NG in CDR-H2 — the same motif, in the same position, as one trastuzumab carries. That matters for how the team reads all of this: trastuzumab has two surviving liabilities, more than CR6261 or b12, and it is an approved drug that works. These ranks compare six molecules to each other, not to therapeutics in general, so they tell us what to measure, not what to kill. The next test I would run is accelerated stress followed by AC-SINS and SEC on CR6261, since its place on the drop list rests on proxies rather than chemistry — if it behaves like the three approved antibodies, it comes off the list.

## 9. Take-home exercises

Due before week 9.

**Exercise 1 — does the threshold matter? (required).**
We called a residue exposed at RSA > 0.25. Recompute the high-risk set at 0.15, 0.25 and 0.40. How much
does the answer move? If your two kill candidates change with the threshold, say so — that is the most
important possible result, and the same robustness check you ran in weeks 3 and 5.

**Exercise 2 — liability versus binding (required).**
For trastuzumab, check whether its high-risk liability residues are in the **paratope** you measured in
week 1. (You will need to convert your week 1 paratope positions to IMGT — week 2's tooling.) If a
liability residue is also a contact residue, what does that do to the idea of simply engineering it out?

**Exercise 3 — the real tool (stretch).**
Submit the panel to [TAP](https://opig.stats.ox.ac.uk/webapps/sabdab-sabpred/sabpred/tap/). Compare its
five metrics and its amber/red flags with your own ranking. Two things to report: whether TAP agrees
with your ordering, and — more interesting — whether any antibody that looks fine against *your* panel
is flagged against TAP's 754-molecule reference. That difference is the value of a real reference set,
measured.

In [ ]:
# Exercise 1 - threshold sensitivity.
# `liabilities` has an `rsa` column; recompute `high_risk` at several cutoffs.

# YOUR CODE HERE
# Exercise 1 - threshold sensitivity.
CUTOFFS = [0.15, 0.25, 0.40]

sens = pd.DataFrame({
    f"RSA>{c}": (scored[(scored.rsa > c) & scored.in_cdr]
                 .groupby("antibody").size())
    for c in CUTOFFS
}).reindex(metrics.antibody).fillna(0).astype(int)
sens["status"] = [sequences[a]["status"] for a in sens.index]

print("High-risk liabilities per antibody, by exposure cutoff:\n")
print(sens.to_string())

print("\nPanel totals:")
for c in CUTOFFS:
    n = int(((scored.rsa > c) & scored.in_cdr).sum())
    print(f"  RSA > {c}:  {n} of {len(scored)} raw hits  ({100*n/len(scored):.0f}%)")

# which motifs appear or vanish between the loosest and strictest cutoff
edge = scored[scored.in_cdr & (scored.rsa > 0.15) & (scored.rsa <= 0.40)]
print(f"\n{len(edge)} hits are threshold-dependent (survive 0.15 but not 0.40):")
print(edge[["antibody", "chain", "kind", "motif", "imgt", "region", "rsa"]]
      .round(2).to_string(index=False))

High-risk liabilities per antibody, by exposure cutoff:

             RSA>0.15  RSA>0.25  RSA>0.4    status
antibody                                          
trastuzumab         2         2        2  approved
pertuzumab          1         1        1  approved
bevacizumab         0         0        0  approved
b12                 2         1        1  research
CR6261              1         1        0  research
murine_4G6K         3         3        2  research

Panel totals:
  RSA > 0.15:  9 of 31 raw hits  (29%)
  RSA > 0.25:  8 of 31 raw hits  (26%)
  RSA > 0.4:  6 of 31 raw hits  (19%)

3 hits are threshold-dependent (survive 0.15 but not 0.40):
   antibody chain          kind motif  imgt region  rsa
        b12     H isomerisation    DS   111   CDR3 0.20
     CR6261     H     oxidation     M   108   CDR3 0.36
murine_4G6K     L     oxidation     M   109   CDR3 0.39


In [ ]:
# Exercise 2 - do liabilities sit in the paratope?
# Week 1 trastuzumab paratope, as positions in the full PDB chain:
PARATOPE_PDB = {"L": [28, 30, 31, 32, 50, 53, 91, 92, 93, 94],
                "H": [33, 50, 52, 57, 58, 59, 99, 103, 104, 105]}

# Convert these to IMGT (week 2), then intersect with the high-risk liability positions.
# YOUR CODE HERE
# Exercise 2 - do liabilities sit in the paratope?
from anarci import anarci

PARATOPE_PDB = {"L": [28, 30, 31, 32, 50, 53, 91, 92, 93, 94],
                "H": [33, 50, 52, 57, 58, 59, 99, 103, 104, 105]}

pdb_id, h_chain, l_chain = PANEL["trastuzumab"][:3]
tz = download(pdb_id)
CHAIN_ID = {"H": h_chain, "L": l_chain}


def pdb_to_imgt(model, chain_id):
    """Map PDB residue number -> IMGT position for one chain."""
    residues = [r for r in model[chain_id] if r.id[0] == " "]
    seq = "".join(index_to_one(three_to_index(r.get_resname()))
                  if r.get_resname() in STANDARD else "X" for r in residues)
    numbering, _, _ = anarci([("q", seq)], scheme="imgt")
    dom_numbering, start, _ = numbering[0][0]
    mapping, i = {}, start
    for (pos, _ins), aa in dom_numbering:
        if aa != "-":
            mapping[residues[i].id[1]] = pos
            i += 1
    return mapping


paratope_imgt = {}
for ch in ("H", "L"):
    m = pdb_to_imgt(tz, CHAIN_ID[ch])
    paratope_imgt[ch] = {m[p] for p in PARATOPE_PDB[ch] if p in m}
    missing = [p for p in PARATOPE_PDB[ch] if p not in m]
    print(f"{ch}: {len(paratope_imgt[ch])} paratope positions mapped"
          + (f"   (unmapped: {missing})" if missing else ""))

tz_hits = scored[(scored.antibody == "trastuzumab") & scored.high_risk]
tz_hits = tz_hits.assign(
    in_paratope=[p in paratope_imgt[c] for c, p in zip(tz_hits.chain, tz_hits.imgt)])
print()
print(tz_hits[["chain", "kind", "motif", "imgt", "region", "rsa", "in_paratope"]]
      .round(2).to_string(index=False))

H: 10 paratope positions mapped
L: 10 paratope positions mapped

chain          kind motif  imgt region  rsa  in_paratope
    H   deamidation    NG    62   CDR2 0.42        False
    H isomerisation    DG   110   CDR3 0.55        False


---

### Where we go next

Developability asks whether a molecule can be *made*. Next week asks whether it can be *given to a
person* without the immune system attacking it — which for a murine antibody like `4G6K` is the
difference between a reagent and a drug.

| Week | Relationship to today |
|---|---|
| **9** | Humanisation and immunogenicity — another early filter, with the same "fixing it may break binding" tension you met in section 5 |
| **10** | Language models score sequences without any structure at all, and can be asked what is *unusual* about an antibody |
| **11** | Redesign — where you will try to remove a liability and watch what it costs |
| **13** | Assembling these filters into one pipeline, and the ways a pipeline lies to you |

Before week 9: finish exercises 1 and 2, and commit.

**One habit from today.** When a screen flags something, ask what it would have said about a molecule
you already know works. Trastuzumab was that control here, and it changed how much weight the output
could carry.